**Improve Performance with Algorithm Tuning**

Having one or two algorithms that perform reasonably well on a problem is a good start, but sometimes you may be incentivised to get the best result you can given the time and resources you have available.

Machine learning algorithms have hyperparameters that allow you to tailor the behavior of the algorithm to your specific dataset.

Hyperparameters are different from parameters, which are the internal coefficients or weights for a model found by the learning algorithm. Unlike parameters, hyperparameters are specified by the practitioner when configuring the model.

Typically, it is challenging to know what values to use for the hyperparameters of a given algorithm on a given dataset, therefore it is common to use random or grid search strategies for different hyperparameter values.

The more hyperparameters of an algorithm that you need to tune, the slower the tuning process. Therefore, it is desirable to select a minimum subset of model hyperparameters to search or tune.

Not all model hyperparameters are equally important. Some hyperparameters have an outsized effect on the behavior, and in turn, the performance of a machine learning algorithm.

As a machine learning practitioner, you must know which hyperparameters to focus on to get a good result quickly.

In [1]:
import pandas as pd
import numpy as np
df=pd.read_csv("diabetes.csv")
filename='diabetes.csv'
names=['preg','glu','bp','skin','insulin','bmi','pedi','age','class']
df=pd.read_csv(filename,names=names)
#separate input and output
X=df.drop('class',axis=True)
Y=df['class']

In [2]:
#Split test and train data
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split (X,Y,test_size=0.3)

In [3]:
X_train

,preg,glu,bp,skin,insulin,bmi,pedi,age
333,12,106,80,0,0,23.6,0.137,44
427,1,181,64,30,180,34.1,0.328,38
600,1,108,88,19,0,27.1,0.400,24
556,1,97,70,40,0,38.1,0.218,30
22,7,196,90,0,0,39.8,0.451,41
...,...,...,...,...,...,...,...,...
1,1,85,66,29,0,26.6,0.351,31
743,9,140,94,0,0,32.7,0.734,45
166,3,148,66,25,0,32.5,0.256,22
454,2,100,54,28,105,37.8,0.498,24


In [4]:
Y_train

333    0
427    1
600    0
556    0
22     1
      ..
1      0
743    1
166    0
454    0
318    0
Name: class, Length: 537, dtype: int64

In [5]:
X_test

,preg,glu,bp,skin,insulin,bmi,pedi,age
7,10,115,0,0,0,35.3,0.134,29
147,2,106,64,35,119,30.5,1.400,34
611,3,174,58,22,194,32.9,0.593,36
649,0,107,60,25,0,26.4,0.133,23
253,0,86,68,32,0,35.8,0.238,25
...,...,...,...,...,...,...,...,...
143,10,108,66,0,0,32.4,0.272,42
55,1,73,50,10,0,23.0,0.248,21
60,2,84,0,0,0,0.0,0.304,21
347,3,116,0,0,0,23.5,0.187,23


In [6]:
Y_test

7      0
147    0
611    1
649    0
253    0
      ..
143    1
55     0
60     0
347    0
275    0
Name: class, Length: 231, dtype: int64

# **Approach 1: Use train_test_split and manually tune parameters by trial and error**

In [7]:
#Design model and fit the model
from sklearn import svm
model=svm.SVC(kernel='rbf',C=30, gamma='auto')
model.fit(X_train,Y_train)
model.score(X_test,Y_test)

0.645021645021645

# **Approach 2: Use K Fold Cross validation**

In [8]:
#cross validation
from sklearn.model_selection import cross_val_score
cross_val_score(svm.SVC(kernel='rbf',C=30, gamma='auto'),X,Y,cv=5)

array([0.64935065, 0.64935065, 0.64935065, 0.65359477, 0.65359477])

In [ ]:
cross_val_score(svm.SVC(kernel='linear',C=20, gamma='auto'),X,Y,cv=5)

array([0.75974026, 0.74675325, 0.75324675, 0.81045752, 0.76470588])

In [ ]:
cross_val_score(svm.SVC(kernel='linear',C=10, gamma='auto'),X,Y,cv=5)

array([0.75974026, 0.75974026, 0.74025974, 0.81045752, 0.76470588])

**Above approach is tiresome and very manual. We can use for loop as an alternative**

In [9]:
import numpy as np
kernels = ['rbf', 'linear']
C = [1,10,20]
avg_scores = {}
for kval in kernels:
    for cval in C:
        cv_scores = cross_val_score(svm.SVC(kernel=kval,C=cval,gamma='auto'),X,Y, cv=5)
        avg_scores[kval + '_' + str(cval)] = np.average(cv_scores)

avg_scores

{'rbf_1': 0.6510482981071216,
 'rbf_10': 0.6510482981071216,
 'rbf_20': 0.6510482981071216,
 'linear_1': 0.7656820303879128,
 'linear_10': 0.7669807316866141,
 'linear_20': 0.7669807316866141}

From above results we can say that rbf with C=10 or 20 or linear with C=10 will give best performance

# **Approach 3: Grid Search Parameter Tuning**

In terms of machine learning, Clf is an estimator instance, which is used to store model. We use clf to store trained model values, which are further used to predict value, based on the previously stored weights. You can check out this tutorial for more information.

In [10]:
from sklearn.model_selection import GridSearchCV
clf = GridSearchCV(svm.SVC(gamma='auto'), {
    'C': [1,10,20],
    'kernel': ['rbf','linear']
}, cv=5, return_train_score=False)
clf.fit(X, Y)
clf.cv_results_

{'mean_fit_time': array([7.29990959e-02, 4.16371555e+00, 3.02196503e-02, 3.38531267e+01,
        2.99761295e-02, 4.10510009e+01]),
 'std_fit_time': array([1.74766370e-02, 1.47055121e+00, 8.20843412e-04, 1.42563858e+01,
        9.45110169e-04, 5.46533399e+00]),
 'mean_score_time': array([0.01951389, 0.00347314, 0.00897155, 0.00374374, 0.00970459,
        0.00359535]),
 'std_score_time': array([0.00682845, 0.00024599, 0.0005683 , 0.00014248, 0.00236349,
        0.00019611]),
 'param_C': masked_array(data=[1, 1, 10, 10, 20, 20],
              mask=[False, False, False, False, False, False],
        fill_value='?',
             dtype=object),
 'param_kernel': masked_array(data=['rbf', 'linear', 'rbf', 'linear', 'rbf', 'linear'],
              mask=[False, False, False, False, False, False],
        fill_value='?',
             dtype=object),
 'params': [{'C': 1, 'kernel': 'rbf'},
  {'C': 1, 'kernel': 'linear'},
  {'C': 10, 'kernel': 'rbf'},
  {'C': 10, 'kernel': 'linear'},
  {'C': 20, 'ker

**Result of GridSearchCV as table**

In [11]:
df = pd.DataFrame(clf.cv_results_)
df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.072999,0.017477,0.019514,0.006828,1,rbf,"{'C': 1, 'kernel': 'rbf'}",0.649351,0.649351,0.649351,0.653595,0.653595,0.651048,0.002079,4
1,4.163716,1.470551,0.003473,0.000246,1,linear,"{'C': 1, 'kernel': 'linear'}",0.759740,0.753247,0.740260,0.810458,0.764706,0.765682,0.023841,3
2,0.030220,0.000821,0.008972,0.000568,10,rbf,"{'C': 10, 'kernel': 'rbf'}",0.649351,0.649351,0.649351,0.653595,0.653595,0.651048,0.002079,4
3,33.853127,14.256386,0.003744,0.000142,10,linear,"{'C': 10, 'kernel': 'linear'}",0.759740,0.759740,0.740260,0.810458,0.764706,0.766981,0.023299,1
4,0.029976,0.000945,0.009705,0.002363,20,rbf,"{'C': 20, 'kernel': 'rbf'}",0.649351,0.649351,0.649351,0.653595,0.653595,0.651048,0.002079,4
5,41.051001,5.465334,0.003595,0.000196,20,linear,"{'C': 20, 'kernel': 'linear'}",0.759740,0.746753,0.753247,0.810458,0.764706,0.766981,0.022564,1


**Statistical comparison of models using grid search**

In [12]:
df[['param_C','param_kernel','mean_test_score']]

,param_C,param_kernel,mean_test_score
0,1,rbf,0.651048
1,1,linear,0.765682
2,10,rbf,0.651048
3,10,linear,0.766981
4,20,rbf,0.651048
5,20,linear,0.766981


**Parameters selection of the best model**

In [13]:
clf.best_params_

{'C': 10, 'kernel': 'linear'}

In [14]:
clf.best_score_

0.7669807316866141

**Directory of estimator instance**

In [15]:
dir(clf)

['__abstractmethods__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_check_feature_names',
 '_check_n_features',
 '_check_refit_for_multimetric',
 '_estimator_type',
 '_format_results',
 '_get_param_names',
 '_get_tags',
 '_more_tags',
 '_pairwise',
 '_repr_html_',
 '_repr_html_inner',
 '_repr_mimebundle_',
 '_required_parameters',
 '_run_search',
 '_select_best_index',
 '_validate_data',
 'best_estimator_',
 'best_index_',
 'best_params_',
 'best_score_',
 'classes_',
 'cv',
 'cv_results_',
 'decision_function',
 'error_score',
 'estimator',
 'feature_names_in_',
 'fit',
 'get_params',
 'inverse_transform',
 'multim

# **Approach 4: Use RandomSearchCV**

Use RandomizedSearchCV to reduce number of iterations and with random combination of parameters. This is useful when you have too many parameters to try and your training time is longer. It helps reduce the cost of computation

In [16]:
from sklearn.model_selection import RandomizedSearchCV
rs = RandomizedSearchCV(svm.SVC(gamma='auto'), {
        'C': [1,10,20],
        'kernel': ['rbf','linear']
    }, 
    cv=5, 
    return_train_score=False, 
    n_iter=2
)
rs.fit(X, Y)
pd.DataFrame(rs.cv_results_)[['param_C','param_kernel','mean_test_score']]

,param_C,param_kernel,mean_test_score
0,20,linear,0.766981
1,1,linear,0.765682


# **How about different models with different hyperparameters?**

In [17]:
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

model_params = {
    'svm': {
        'model': svm.SVC(gamma='auto'),
        'params' : {
            'C': [1,10,20],
            'kernel': ['rbf','linear']
        }  
    },
    'random_forest': {
        'model': RandomForestClassifier(),
        'params' : {
            'n_estimators': [1,5,10]
        }
    },
    'logistic_regression' : {
        'model': LogisticRegression(solver='liblinear',multi_class='auto'),
        'params': {
            'C': [1,5,10]
        }
    }
}

In [18]:
scores = []

for model_name, mp in model_params.items():
    clf =  GridSearchCV(mp['model'], mp['params'], cv=5, return_train_score=False)
    clf.fit(X, Y)
    scores.append({
        'model': model_name,
        'best_score': clf.best_score_,
        'best_params': clf.best_params_
    })
    
df = pd.DataFrame(scores,columns=['model','best_score','best_params'])
df

,model,best_score,best_params
0,svm,0.766981,"{'C': 10, 'kernel': 'linear'}"
1,random_forest,0.756532,{'n_estimators': 10}
2,logistic_regression,0.768288,{'C': 5}


Based on above, I can conclude that SVM with C=10 and kernel='linear' is the best model for solving my problem of diabetes classification